# 06 - Exploratory: Isolation Forest Anomaly Safety Net

**Notebook version:** v18 -- 2026-07-29

- Fit Isolation Forest on the full 591-feature set (unsupervised, no label leakage possible)
- Score all wafers; flag anomalies independent of the reduced supervised model
- Cross-check: does the anomaly detector catch fails the reduced model's false negatives?
- Report as an exploratory robustness finding, not a formal RQ (src/anomaly_detection.py)


In [ ]:
# --- Colab setup: run this cell first if you opened this notebook from GitHub in Colab ---
# If you're running locally in Jupyter from the notebooks/ folder, this cell does nothing.
# Safe to re-run: always anchors to /content so repeated runs never create nested clones.
# git pull always runs (cheap, ~seconds) so code is never stale even if Colab's
# 'Restart session' left /content on disk from an earlier session -- only
# pip install (the actual slow part) is skipped via the session marker.
import os
import subprocess
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/WJPsystems/secom-explainable-vm.git"
REPO_NAME = "secom-explainable-vm"
SETUP_MARKER = Path("/content/.secom_setup_done")

if IN_COLAB:
    os.chdir("/content")
    if not os.path.exists(REPO_NAME):
        !git clone "{REPO_URL}"
        SETUP_MARKER.unlink(missing_ok=True)  # fresh clone -- force full setup below

    # Always pull -- cheap, and guarantees code is current even if /content
    # persisted on disk from an earlier session (e.g. Colab 'Restart session'
    # rather than a full 'Disconnect and delete runtime').
    os.chdir(f"/content/{REPO_NAME}")
    !git pull
    os.chdir("/content")

    os.chdir(f"/content/{REPO_NAME}/notebooks")

    already_setup = SETUP_MARKER.exists()
    if not already_setup:
        !pip install -q -r ../requirements.txt
        SETUP_MARKER.touch()
        setup_note = "Ran pip install (git pull always runs regardless)."
    else:
        setup_note = "Skipped pip install -- already done earlier this session. git pull always ran above."

    commit_info = subprocess.run(
        ["git", "log", "-1", "--format=%h %ci"], capture_output=True, text=True
    ).stdout.strip()
    print(f"Colab setup complete. Working directory: {os.getcwd()}")
    print(setup_note)
    print(f"Repo commit: {commit_info}")
    print("Compare this commit hash against GitHub's latest commit to confirm you're current.")
else:
    print("Not running in Colab -- assuming local Jupyter launched from the notebooks/ folder.")

In [ ]:
import sys

try:
    import google.colab
    _SRC_PATH = "/content/secom-explainable-vm/src"
except ImportError:
    _SRC_PATH = "../src"
if _SRC_PATH not in sys.path:
    sys.path.append(_SRC_PATH)

import numpy as np
import pandas as pd

from preprocessing import load_raw, screen_missingness, screen_variance, impute_median
from artifacts import load_json, load_model
from anomaly_detection import fit_anomaly_detector, score_wafers, compare_against_supervised_flags

X, y = load_raw()
X = impute_median(screen_variance(screen_missingness(X)))
print(f"Screened feature count: {X.shape[1]}")


## Step 1: Load RQ1/RQ3 artifacts

Uses the SAME held-out split as RQ2/RQ3 (`rq1_holdout_split`) and RQ3's
actual reduced feature set + tuned threshold (`rq3_summary`) -- not
re-derived assumptions, real saved outputs from those notebooks.

In [ ]:
holdout_split = load_json("rq1_holdout_split")
train_idx, test_idx = holdout_split["train_idx"], holdout_split["test_idx"]
X_train_ho, X_test_ho = X.iloc[train_idx], X.iloc[test_idx]
y_train_ho, y_test_ho = y.iloc[train_idx], y.iloc[test_idx]

rq3_summary = load_json("rq3_summary")
final_features = rq3_summary["final_features"]
tuned_threshold = rq3_summary["tuned_threshold"]
print(f"Reduced feature set ({len(final_features)}): {final_features}")
print(f"Tuned threshold (from RQ3): {tuned_threshold:.4f}")


## Step 2: Fit the Isolation Forest on the FULL 432-feature set

Deliberately trained on all 432 screened features, not the reduced 10 --
the entire point of this safety net (see docs/synopsis.docx, "Robustness
Consideration") is catching a novel failure mode that the reduced,
label-driven feature set might not capture. Unsupervised, so it never
sees the label at all -- fit only on the training portion of the holdout
split, scored on the held-out test portion, same discipline as everywhere
else in this pipeline.

In [ ]:
anomaly_model = fit_anomaly_detector(X_train_ho, contamination=0.066, random_state=42)
anomaly_df = score_wafers(anomaly_model, X_test_ho)

n_flagged = anomaly_df["is_anomaly"].sum()
print(f"Isolation Forest flagged {n_flagged} of {len(X_test_ho)} held-out wafers as anomalous "
      f"({n_flagged/len(X_test_ho):.1%})")


## Step 3: Compare against the supervised reduced model at its tuned threshold

Retrains RQ3's reduced-feature model architecture (from RQ1's best tree
type) on the train portion, applies RQ3's actual tuned threshold (not the
default 0.5, which we already know collapses to predicting "pass" for
everyone) to get real supervised predictions to compare against.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

rq1_summary = load_json("rq1_summary")
best_tree_name = rq1_summary["best_tree_model_name"]
pos_weight = (y_train_ho == 0).sum() / (y_train_ho == 1).sum()

if best_tree_name == "XGBoost":
    reduced_model = XGBClassifier(
        n_estimators=200, max_depth=4, scale_pos_weight=pos_weight,
        eval_metric="logloss", random_state=42, n_jobs=-1,
    )
elif best_tree_name == "RandomForest":
    reduced_model = RandomForestClassifier(
        n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1,
    )
else:
    raise ValueError(f"Unexpected best_tree_model_name: {best_tree_name}")

reduced_model.fit(X_train_ho[final_features], y_train_ho)
reduced_probs = reduced_model.predict_proba(X_test_ho[final_features])[:, 1]
reduced_preds_tuned = pd.Series((reduced_probs >= tuned_threshold).astype(int), index=X_test_ho.index)

comparison = compare_against_supervised_flags(anomaly_df, y_test_ho, reduced_preds_tuned)
print("Comparison (Isolation Forest vs. supervised reduced model at tuned threshold):")
for k, v in comparison.items():
    print(f"  {k}: {v}")

print(
    "\nInterpretation: 'supervised_false_negatives' is how many real fails the "
    "reduced model missed even at its tuned threshold. "
    "'of_those_flagged_by_anomaly_detector' is how many of those the unsupervised "
    "safety net still caught -- this is the concrete, measurable value of running "
    "both in parallel rather than relying on the supervised model alone."
)


## Next steps

- Report the anomaly-detector flag rate and the false-negative recovery
  rate in the capstone write-up as this exploratory robustness check's
  deliverable
- Not a formal RQ -- reported as a secondary, exploratory finding per the
  synopsis's "Robustness Consideration" section

## Export

Run the cell below last to export this notebook to a standalone HTML file.

In [ ]:
# --- Export this notebook to HTML (run this cell last) ---
# Works whether opened live from GitHub in Colab or run locally in Jupyter.
import json
import subprocess

NOTEBOOK_NAME = "06_anomaly_safety_net"

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

export_path = f"{NOTEBOOK_NAME}.ipynb"
live_export_available = False

if IN_COLAB:
    # Colab's own notebook JSON isn't the same file as the clone on disk --
    # this pulls the live, currently-run state (including your outputs)
    # directly from the Colab frontend, so nothing gets missed. This only
    # works when there's an actual live frontend attached (i.e. you're running
    # this cell interactively yourself) -- it returns None instead of raising
    # when run unattended (e.g. via 00_run_all.ipynb's automated execution),
    # so that case is caught explicitly here rather than left to crash with a
    # raw TypeError, which used to make 00_run_all's --allow-errors flag mask
    # *real* failures elsewhere in the notebook, not just this expected one.
    from google.colab import _message
    response = _message.blocking_request('get_ipynb', timeout_sec=30)
    if response is not None:
        ipynb_content = response['ipynb']
        with open(export_path, 'w') as f:
            json.dump(ipynb_content, f)
        live_export_available = True
    else:
        print(
            "No live Colab frontend detected (expected when run via "
            "00_run_all.ipynb) -- skipping the live export. 00_run_all does its "
            "own separate HTML export against the already-executed file instead."
        )

if live_export_available or not IN_COLAB:
    html_output = f"{NOTEBOOK_NAME}.html"
    result = subprocess.run(
        ['jupyter', 'nbconvert', '--to', 'html', export_path, '--output', html_output],
        capture_output=True, text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
    else:
        print(f"Exported to {html_output}")

    if IN_COLAB and result.returncode == 0:
        from google.colab import files
        files.download(html_output)
